In [1]:
import pandas as pd
from pathlib import Path
from datetime import datetime

In [2]:
PROJECT_DIR = Path("/home/sc4r3cr0w/Schreibtisch/Data Analytics/WBS_Final_Project")
RAW = PROJECT_DIR / "RAW"
CLEAN_DIR = PROJECT_DIR / "CLEAN"
CLEAN_DIR.mkdir(exist_ok=True)

INCOME_DIR = RAW / "income"

print(INCOME_DIR.exists(), len(list(INCOME_DIR.glob('*'))))

True 2


In [3]:
def read_workplace_income(file):
    """Read sheet 8.2 (median income by workplace) and return one row per district and year."""
    df = pd.read_excel(file, sheet_name="8.2", header=None)

    date_cols = {}
    for col in df.columns:
        for value in df[col]:
            if isinstance(value, datetime):
                date_cols[col] = value.year
                break

    is_district = df[0].astype(str).str.match(r"^\d{5}$")
    is_total = df[2] == "Insgesamt"
    districts = df[is_district & is_total]

    frames = []
    for col, reference_year in date_cols.items():
        frame = pd.DataFrame({
            "year": reference_year + 1,
            "district_key": districts[0],
            "district_name": districts[1],
            "income_median_workplace": districts[col],
        })
        frames.append(frame)

    return pd.concat(frames, ignore_index=True)

In [4]:
workplace_2020 = read_workplace_income(INCOME_DIR / "entgelt-d-0-202012-xlsx.xlsx")
workplace_2025 = read_workplace_income(INCOME_DIR / "entgelt-dwolk-0-202512-xlsx.xlsx")

income_workplace = pd.concat([workplace_2020, workplace_2025], ignore_index=True)
income_workplace["income_median_workplace"] = pd.to_numeric(income_workplace["income_median_workplace"])

print("Rows:", len(income_workplace))
print("Duplicates:", income_workplace.duplicated(subset=["year", "district_key"]).sum())
income_workplace.groupby("year")["income_median_workplace"].agg(["count", "median"]).round(0)

Rows: 4010
Duplicates: 0


,count,median
year,,
2017,401,2992.0
2018,401,3070.0
2019,401,3170.0
2020,401,3251.0
2021,401,3259.0
2022,401,3344.0
2023,401,3466.0
2024,401,3604.0
2025,401,3799.0


In [5]:
income_workplace.to_csv(CLEAN_DIR / "income_workplace.csv", index=False)
print("Saved:", CLEAN_DIR / "income_workplace.csv", "| rows:", len(income_workplace))

Saved: /home/sc4r3cr0w/Schreibtisch/Data Analytics/WBS_Final_Project/CLEAN/income_workplace.csv | rows: 4010
